In [75]:
import torch
from torch.utils import data
from torchvision import transforms as T

from src.dataloaders.dataloader_for_CNN import mavDataLoader, mavDatasetCNN_3D, SequenceBatchSampler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Обучение на:', device, sep=' ')

transform = T.Compose([
    T.Resize((320, 192))
])
dataset = mavDatasetCNN_3D('datasets/euroc_mav', transform, device='cpu', batchsize=32, hidden_size=0, lst_of_datasets=['mav0_easy1']) # Сразу формируем все массивы на GPU

groups = dataset.batch_groups.copy()

train_size = int(0.8 * len(groups))
train_groups = groups[:train_size]
test_groups = groups[train_size:]


train_sampler = SequenceBatchSampler(train_groups)
test_sampler = SequenceBatchSampler(test_groups)

train_data = data.DataLoader(dataset, batch_sampler=train_sampler, num_workers=6, pin_memory=True)
test_data = data.DataLoader(dataset, batch_sampler=test_sampler, num_workers=6, pin_memory=True)

Обучение на: cuda


In [76]:
len(train_data)

908

In [77]:
len(test_data)

228

In [78]:
from src.geometry.RotationTorch import RotationTorch as RT 
from src.geometry.PoseTorch import PoseTorch as PT  
from src.geometry.TrajectoryTorch import TrajectoryTorch as TT

from src.function_of_loss.mse_pose import PoseLoss

In [23]:
import torch.nn as nn

class PoseLoss(nn.Module):
    def __init__(self, k: float = 100, reduction: str = 'mean'):
        super().__init__()
        self.k = k # весовой коэф. для балансировки вкладов перемещения и вращения
        self.mse = nn.MSELoss(reduction=reduction)
        
    def forward(self, y_pred: torch.Tensor, y_fact: torch.Tensor):
        
        pred_p = y_pred[..., :3]
        pred_eul = y_pred[..., 3:]
        
        fact_p = y_fact[..., :3]
        fact_eul = y_fact[..., 3:]
        
        self.pos_loss = self.mse(pred_p, fact_p)
        self.r_loss = self.mse(pred_eul, fact_eul)
        
        return self.pos_loss + self.k * self.r_loss

In [79]:
dt = iter(train_data)
x, y, T_m = next(dt)
x2, y2, T_m2 = next(dt)
x3, y3, T_m3 = next(dt)
x4, y4, T_m4 = next(dt)

Y1 = torch.stack([y, y2])
Y2 = torch.stack([y3, y4])

In [107]:
y[0]

tensor([ 3.7278e-03, -2.9076e-06, -1.4995e-03,  1.0389e-03, -1.5391e-03,
         6.1206e-04], dtype=torch.float64)

In [100]:
Y, Y2 = PT.from_lie(y), PT.from_lie(y2)

torch.linalg.norm((Y.inv() * Y2).as_lie()[0])

tensor(0.0014, dtype=torch.float64)

In [106]:
loss_func = PoseLoss(reduction='sum')
loss_func(y3, y4)


tensor(0.0109, dtype=torch.float64)

In [95]:
y[0]

tensor([ 3.7278e-03, -2.9076e-06, -1.4995e-03,  1.0389e-03, -1.5391e-03,
         6.1206e-04], dtype=torch.float64)

In [82]:
l = TT.from_lie_relative(y3, PT.from_lie(T_m3)[0]).path_length()
100 * loss_func.r_loss / l * 100, loss_func.pos_loss / l * 100

(tensor(0.1542, dtype=torch.float64), tensor(0.0026, dtype=torch.float64))

In [83]:
from src.models.CNN_ResNet50_VO import CNN_ResNet50_VO

model_cnn = CNN_ResNet50_VO()
state_dict_cnn = torch.load('process_of_fitting/fitting_models/CNNResNet50_VO_2_0.tar')
model_cnn.load_state_dict(state_dict_cnn)
model_cnn = model_cnn.to(device)

In [85]:
model_cnn.eval()
with torch.no_grad():
    y_p = model_cnn(x.to(device))
    
y_p.shape

torch.Size([32, 6])

In [93]:
loss_func(y_p, y3.to(device))
100 * loss_func.r_loss / l * 100, loss_func.pos_loss / l * 100

(tensor(0.1804, device='cuda:0', dtype=torch.float64),
 tensor(0.0043, device='cuda:0', dtype=torch.float64))